In [1]:
from langchain_community.document_loaders import TextLoader

# Load all recipe files
file_paths = ["Spaghetti Carbonara.txt", "Classic Pancakes.txt", "Chicken Curry.txt"]
documents = []

for file_path in file_paths:
    loader = TextLoader(file_path)
    docs = loader.load()
    documents.extend(docs)

print(f"Loaded {len(documents)} documents")

Loaded 3 documents


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Use a free HuggingFace model for embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create FAISS vector store from documents
vectorstore = FAISS.from_documents(documents, embeddings)

print(f"Number of documents in vector store: {vectorstore.index.ntotal}")

# Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

Number of documents in vector store: 3


In [3]:
from langchain.tools import tool
import re

# =============================
# Recipe Parsing & Scaling
# =============================
def parse_recipe(recipe_text: str):
    sections = {"title": "", "servings": 1, "ingredients": [], "instructions": []}
    current_section = None

    for line in recipe_text.splitlines():
        line = line.strip()
        if not line:
            continue

        # Detect section headers
        if line.lower() == "title":
            current_section = "title"
            continue
        elif line.lower().startswith("ingredients"):
            current_section = "ingredients"
            continue
        elif line.lower().startswith("instructions"):
            current_section = "instructions"
            continue

        # Detect serving size (e.g., "Serving: 1 person")
        if line.lower().startswith("serving"):
            match = re.search(r"(\d+)", line)
            if match:
                sections["servings"] = int(match.group(1))
            continue

        # Assign lines to sections
        if current_section == "title":
            sections["title"] = line
        elif current_section == "ingredients":
            sections["ingredients"].append(line)
        elif current_section == "instructions":
            sections["instructions"].append(line)

    return sections

def scale_ingredients(ingredients, scale_factor):
    scaled_ingredients = []
    for item in ingredients:
        numbers = re.findall(r"\d*\.?\d+", item)
        if numbers:
            scaled_item = item
            for num in numbers:
                scaled_value = float(num) * scale_factor
                scaled_item = scaled_item.replace(num, str(round(scaled_value, 2)), 1)
            scaled_ingredients.append(scaled_item)
        else:
            scaled_ingredients.append(item)
    return scaled_ingredients

# LangChain Tool for scaling
@tool
def scaler_tool(recipe_text: str, servings: int):
    """
    Scales a recipe to the given number of servings.
    Arguments:
        recipe_text (str): Raw recipe text.
        servings (int): Desired servings.
    Returns:
        dict: JSON with scaled ingredients and instructions.
    """
    parsed = parse_recipe(recipe_text)
    original_servings = parsed.get("servings", 1)
    scale_factor = servings / original_servings  # ✅ Correct multiplier
    scaled_ingredients = scale_ingredients(parsed["ingredients"], scale_factor)
    parsed["ingredients"] = scaled_ingredients
    parsed["servings"] = servings  # Update servings in output
    return parsed


In [4]:
#SETUP FOR MODEL
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4.1-mini", model_provider="openai")
# Bind tools to the model
model_with_tools = model.bind_tools([scaler_tool])


Enter API key for OpenAI:  ········


In [5]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langchain_core.messages import SystemMessage

workflow = StateGraph(state_schema=MessagesState) #creation of graph and setting scheme

# --- NODE: Retriever ---
def retrieve_docs(state: MessagesState):
    user_message = state["messages"][-1].content # Get the last user query from the state
    docs = retriever.invoke(user_message) # Retrieve relevant docs
    context_text = "\n\n".join([doc.page_content for doc in docs]) # Combine docs into a single context string
    context_message = f"Use the following context to answer:\n\n{context_text}" # Add context to the state (prepend as a system message)
    return {"messages": [SystemMessage(content=context_message)]}

# --- NODE: Model ---
def call_model(state: MessagesState):
    # Call the model with tool binding
    response = model_with_tools.invoke(state["messages"])

    # Check if the model called a tool
    if getattr(response, "tool_calls", None):
        for tool_call in response.tool_calls:
            if tool_call["name"] == "scaler_tool":
                # Execute the scaler tool
                tool_result = scaler_tool.invoke(tool_call["args"])

                # Add the tool result back into conversation as a system message
                state["messages"].append(SystemMessage(content=f"Tool result:\n{tool_result}"))

                # Re-run the model with updated messages
                response = model_with_tools.invoke(state["messages"])

    return {"messages": response}

# --- Add nodes to graph ---
workflow.add_node("retriever", retrieve_docs)
workflow.add_node("model", call_model)

# --- Add edges for flow ---
workflow.add_edge(START, "retriever")
workflow.add_edge("retriever", "model")

# --- Compile graph ---
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)#checkpointer is a parameter expected by langraph

In [6]:
config = {"configurable": #reserved keyword in langraph
          {"thread_id": "abc123"} #custom key to track session
         }

In [9]:
# Import necessary modules
from rich.console import Console
from rich.panel import Panel
from rich.align import Align
from langchain_core.messages import HumanMessage

console = Console()

console.print("\t\t[bold green]====== Chat Started (type 'exit' to quit) ======[/bold green]")

while True:
    # User input
    user_input = input("Enter Prompt:")
    if user_input.strip().lower() in ["exit", "quit"]:
        console.print("[bold red]Chat ended.[/bold red]")
        break

    # Display user message in a styled panel
    console.print(Panel(Align.left(user_input), title="[bold blue]You[/bold blue]", border_style="blue"))

    # Call the LangGraph workflow
    response = app.invoke({"messages": [HumanMessage(content=user_input)]}, config)

    # Extract last AI message
    last_msg = response["messages"][-1]
    if last_msg.type == "ai":
        console.print(Panel(Align.left(last_msg.content), title="[bold magenta]AI[/bold magenta]", border_style="magenta"))


====== Chat Started (type 'exit' to quit) ======

Enter Prompt: whats my name


╭────────────────────────────────────────────────────── You ──────────────────────────────────────────────────────╮
│ whats my name                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── AI ───────────────────────────────────────────────────────╮
│ I don't have information about your name. How can I assist you further?                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Enter Prompt: nofel is my name give spegati recipe for 2 people


...

Enter Prompt: whars my name


╭────────────────────────────────────────────────────── You ──────────────────────────────────────────────────────╮
│ whars my name                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── AI ───────────────────────────────────────────────────────╮
│ You mentioned earlier that your name is Nofel. How can I assist you further?                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Enter Prompt: exit


Chat ended.